# Lab 43 — QML Teórico: PAC Learning, VC Dimension y Límites Fundamentales

Este laboratorio estudia los fundamentos teóricos del aprendizaje automático cuántico (QML):

- Parte A: PAC learning y complejidad de muestra cuántica
- Parte B: Dimensión VC y capacidad del modelo
- Parte C: Barren plateaus — análisis teórico con gradientes
- Parte D: Dequantization — cuándo puede un algoritmo clásico igualar a uno cuántico

**Referencias:**
- Shalev-Shwartz & Ben-David: *Understanding Machine Learning* (Capítulos 2-6)
- Cerezo et al. (2021): *Variational Quantum Algorithms*, Nature Reviews Physics
- Tang (2019): *A Quantum-Inspired Classical Algorithm for Recommendation Systems*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from scipy.optimize import minimize

print('Entorno QML teórico listo')

## Parte A: PAC Learning Cuántico

El modelo PAC (Probably Approximately Correct) cuantifica cuántas muestras necesita un algoritmo para aprender con alta probabilidad.

**Teorema fundamental del aprendizaje PAC** (para hipótesis finita H):
$$m \geq \frac{1}{\epsilon}\left(\ln|H| + \ln\frac{1}{\delta}\right) \Rightarrow \Pr[\text{error} \leq \epsilon] \geq 1-\delta$$

Para clases infinitas (parametrizadas por VC-dim d):
$$m = O\left(\frac{d + \ln(1/\delta)}{\epsilon}\right)$$

**En QML:** el problema central es que codificar n datos clásicos requiere O(n) operaciones cuánticas (bottleneck de carga de datos).

In [ ]:
def pac_sample_complexity_finite(H_size: int, epsilon: float, delta: float) -> int:
    """Complejidad de muestra PAC para hipótesis finita |H|."""
    return int(np.ceil((1 / epsilon) * (np.log(H_size) + np.log(1 / delta))))


def pac_sample_complexity_vc(vc_dim: int, epsilon: float, delta: float) -> int:
    """Cota de Vapnik-Chervonenkis para clase de hipótesis de dimensión vc_dim."""
    # Cota superior de Blumer et al. 1989
    return int(np.ceil((4 / epsilon) * (2 * vc_dim * np.log(8 * vc_dim / epsilon) + np.log(4 / delta))))


# Visualizar complejidad de muestra vs epsilon
epsilons = np.logspace(-2, -0.3, 50)
delta = 0.05

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel izquierdo: PAC finito vs tamaño de hipótesis
for log_H in [2, 4, 8, 16]:
    m_vals = [pac_sample_complexity_finite(2**log_H, eps, delta) for eps in epsilons]
    axes[0].loglog(epsilons, m_vals, label=f'|H|=2^{log_H}')

axes[0].set_xlabel('ε (error target)')
axes[0].set_ylabel('Muestras necesarias m')
axes[0].set_title('PAC Learning — Clases finitas')
axes[0].legend()
axes[0].invert_xaxis()

# Panel derecho: VC dimension
for d in [2, 5, 10, 20]:
    m_vals = [pac_sample_complexity_vc(d, eps, delta) for eps in epsilons]
    axes[1].loglog(epsilons, m_vals, label=f'VC-dim={d}')

axes[1].set_xlabel('ε (error target)')
axes[1].set_ylabel('Muestras necesarias m')
axes[1].set_title('PAC Learning — Cota VC dimension')
axes[1].legend()
axes[1].invert_xaxis()

plt.suptitle(f'Complejidad de muestra PAC (δ={delta})', y=1.02)
plt.tight_layout()
plt.show()

# Ejemplo concreto
print('Ejemplo: aprender con ε=5%, δ=5%')
print(f'  Clase finita |H|=2^10: m ≥ {pac_sample_complexity_finite(2**10, 0.05, 0.05)}')
print(f'  VC-dim=10:             m ≥ {pac_sample_complexity_vc(10, 0.05, 0.05)}')

## Parte B: Dimensión VC de Circuitos Cuánticos Parametrizados

Para un circuito cuántico parametrizado con $n$ parámetros medibles en el eje Z:

$$\text{VC-dim}(C) = O(n_\theta \cdot n_\text{qubits})$$

Haug & Kim (2021) demostraron que la VC-dim de un QNN de $p$ parámetros es $\Theta(p)$, similar a redes clásicas.

In [ ]:
def build_pqc(n_qubits: int, n_layers: int) -> QuantumCircuit:
    """Construye un PQC (Parametrized Quantum Circuit) de n capas HEA."""
    n_params = n_qubits * n_layers * 2  # Ry + Rz por qubit por capa
    params = ParameterVector('θ', n_params)
    qc = QuantumCircuit(n_qubits)

    idx = 0
    for _ in range(n_layers):
        for q in range(n_qubits):
            qc.ry(params[idx], q)
            qc.rz(params[idx + 1], q)
            idx += 2
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)

    return qc


# Capacidad expresiva vs número de parámetros
configs = [(n, l) for n in [2, 3, 4] for l in [1, 2, 3, 4]]
n_params_list = []
vc_upper_list = []

for n, l in configs:
    qc = build_pqc(n, l)
    n_params = qc.num_parameters
    vc_upper = n_params  # VC-dim ~ n_params (resultado teórico)
    n_params_list.append(n_params)
    vc_upper_list.append(vc_upper)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(n_params_list, vc_upper_list, c='steelblue', s=60, zorder=5)
xs = np.array(sorted(set(n_params_list)))
ax.plot(xs, xs, 'r--', label='VC-dim = n_params (cota lineal)')
ax.set_xlabel('Número de parámetros del PQC')
ax.set_ylabel('VC-dim (cota superior)')
ax.set_title('VC Dimension de Circuitos Cuánticos Parametrizados')
ax.legend()
plt.tight_layout()
plt.show()

print('VC-dim escala linealmente con el número de parámetros,')
print('como en redes neuronales clásicas — QML no tiene ventaja expresiva inherente.')

## Parte C: Barren Plateaus — Análisis Teórico

El gradiente de un circuito variacional con función de costo global tiene varianza exponencialmente pequeña:

$$\text{Var}\left[\frac{\partial C}{\partial \theta_k}\right] \leq \frac{c}{4^n}$$

**Mitigación: inicialización identity-block**
- Inicializar bloques del ansatz como identidades: ∂C/∂θ puede ser O(1/poly(n))
- Usar funciones de costo locales en lugar de globales

In [ ]:
def gradient_variance(n_qubits: int, n_samples: int = 200) -> float:
    """Estima Var[∂C/∂θ_0] para un PQC aleatorio de n_qubits."""
    observable = SparsePauliOp.from_list([('Z' * n_qubits, 1.0)])  # ZZ...Z global
    estimator = StatevectorEstimator()
    gradients = []

    n_params = 2 * n_qubits  # una capa
    eps = 0.01

    for _ in range(n_samples):
        theta = np.random.uniform(0, 2 * np.pi, n_params)

        # Calcular gradiente por diferencias finitas en θ_0
        params = ParameterVector('θ', n_params)
        qc = QuantumCircuit(n_qubits)
        for q in range(n_qubits):
            qc.ry(params[2*q], q)
            qc.rz(params[2*q+1], q)
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)

        theta_p = theta.copy(); theta_p[0] += eps
        theta_m = theta.copy(); theta_m[0] -= eps

        job_p = estimator.run([(qc, observable, [theta_p])])
        job_m = estimator.run([(qc, observable, [theta_m])])

        E_p = job_p.result()[0].data.evs
        E_m = job_m.result()[0].data.evs
        grad = (E_p - E_m) / (2 * eps)
        gradients.append(float(grad))

    return float(np.var(gradients))


# Medir varianza de gradiente para diferentes tamaños de sistema
qubit_range = range(2, 8)
variances = []

print('Calculando varianza de gradiente...')
for n in qubit_range:
    var = gradient_variance(n, n_samples=100)
    variances.append(var)
    print(f'  n={n}: Var[grad] = {var:.2e}')

# Ajuste exponencial
from scipy.optimize import curve_fit
def exp_fit(x, a, b):
    return a * np.exp(b * x)

n_arr = np.array(list(qubit_range))
try:
    popt, _ = curve_fit(exp_fit, n_arr, variances, p0=[variances[0], -0.5])
    fit_label = f'Ajuste: {popt[0]:.3f}·e^({popt[1]:.3f}n)'
    theoretical = [4**(-n) * variances[0] * 4**n_arr[0] for n in n_arr]
except Exception:
    fit_label = 'Ajuste no convergió'
    theoretical = variances

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(n_arr, variances, 'b-o', ms=6, label='Var[grad] medido')
ax.semilogy(n_arr, [4**(-n) for n in n_arr], 'r--', label='Teoría: 4^(-n)')
ax.set_xlabel('Número de qubits n')
ax.set_ylabel('Var[∂C/∂θ₀] (escala log)')
ax.set_title('Barren Plateaus: Varianza del gradiente vs tamaño del sistema')
ax.legend()
plt.tight_layout()
plt.show()

## Parte D: Dequantization y Límites del QML

Tang (2019) mostró que el algoritmo cuántico de recomendaciones (Kerenidis-Prakash) puede ser *dequantizado*: un algoritmo clásico aleatorizado puede lograr la misma complejidad asintótica con acceso a una estructura de datos de muestreo de bajo rango.

**Condición para ventaja cuántica genuina en QML:**
- Los datos deben estar en superposición cuántica (no solo en RAM)
- El acceso a los datos debe ser cuántico (QRAM)
- La tarea de salida debe ser cuántica (no clasicamente verificable)

**Jerarquía de complejidad QML:**
| Tarea | Ventaja cuántica | Estado |
|-------|-----------------|--------|
| HHL sistemas lineales | Exponencial (con QRAM) | Condicionada a QRAM |
| QAE Monte Carlo | Cuadrática O(1/ε) vs O(1/ε²) | Demostrada |
| QML clasificación | Ninguna demostrada | Activo |
| Quantum kernel SVM | Potencial (datos cuánticos) | Por demostrar |

In [ ]:
# Ilustrar el speedup cuántico en QAE vs Monte Carlo clásico
def classical_mc_samples(epsilon: float) -> int:
    """Muestras necesarias para Monte Carlo clásico con error ε: O(1/ε²)."""
    return int(np.ceil(1 / epsilon**2))


def quantum_ae_samples(epsilon: float) -> int:
    """Consultas necesarias para QAE con error ε: O(1/ε)."""
    return int(np.ceil(1 / epsilon))


epsilons = np.logspace(-3, -0.5, 50)
mc_samples = [classical_mc_samples(eps) for eps in epsilons]
qae_samples = [quantum_ae_samples(eps) for eps in epsilons]
speedup = [mc / qae for mc, qae in zip(mc_samples, qae_samples)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].loglog(epsilons, mc_samples, 'r-', lw=2, label='Monte Carlo: O(1/ε²)')
axes[0].loglog(epsilons, qae_samples, 'b-', lw=2, label='QAE: O(1/ε)')
axes[0].invert_xaxis()
axes[0].set_xlabel('ε (precisión)')
axes[0].set_ylabel('Consultas/muestras')
axes[0].set_title('QAE vs Monte Carlo Clásico')
axes[0].legend()

axes[1].loglog(epsilons, speedup, 'g-', lw=2)
axes[1].invert_xaxis()
axes[1].set_xlabel('ε (precisión)')
axes[1].set_ylabel('Speedup (MC / QAE)')
axes[1].set_title('Speedup cuántico de QAE: O(1/ε)')
axes[1].fill_between(epsilons, 1, speedup, alpha=0.3, color='green')

plt.suptitle('Ventaja cuántica demostrada: Quantum Amplitude Estimation')
plt.tight_layout()
plt.show()

print('Para ε = 1%:')
print(f'  MC clásico necesita {classical_mc_samples(0.01):,} muestras')
print(f'  QAE necesita       {quantum_ae_samples(0.01):,} consultas')
print(f'  Speedup: {classical_mc_samples(0.01) / quantum_ae_samples(0.01):.0f}x')

## Resumen

**Lecciones clave del QML teórico:**

1. **PAC learning:** Los circuitos cuánticos tienen complejidad de muestra comparable a redes clásicas (VC-dim ∝ n_params)

2. **Barren plateaus:** El gradiente decrece exponencialmente con el número de qubits para ansätze globales — limita la escala práctica

3. **Dequantization:** Muchas ventajas QML propuestas pueden ser igualdas clásicamente con acceso eficiente a los datos

4. **Ventajas reales:** QAE logra speedup cuadrático genuino. Ventajas QML para datos clásicos generales aún no demostradas

5. **Camino futuro:** QML tiene potencial para datos nativamente cuánticos (tomografía, química, physics-informed)